In [1]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import os
from ast import literal_eval
from collections import Counter
import math
import numpy as np

In [2]:
username = os.path.expanduser('~')
username

'/home/ehdgus7279'

In [15]:
benchmarks = [
    # Low-sharing Apps
    "fdtd2d", "jacobi1d", "fastwalshtransform", "relu", "fft", "fir", 
    "sssp", "convolution2d", "color", "bitonicsort", "pagerank", "simpleconvolution",
    # High-sharing Apps
    'gups', 'syr2k', 'spmv', 'mis', 'stencil2d', 'gemm', 'lu', 
    'matrixmultiplication', 'mm2', 'mm3', 'gaussian', 'matrixtranspose', 
    'im2col', 'vgg16', 'bfs',
]

In [16]:
num_chiplets = 36
ns = 0.000000001

In [17]:
base_dir=f'{username}/mgvm/results/reports'
sub_dir = "mesh"
platform = "mesh"
report_dir = f'{base_dir}/{sub_dir}'
print(report_dir)

/home/ehdgus7279/mgvm/results/reports/mesh


In [18]:
high_central = [14, 25, 20, 21]
low_central = [7,8,9,10,
                13,16,
                19,22,
                25,28]
edge = [1,2,3,4,
        6,12,18,24,
        11,17,23,29,
        31,32,33,34]
corner = [0, 5, 30, 35]

columns = ['benchmark', 
           'high_central',
           'low_central',
           'edge',
           'corner',
           ]

result_df = pd.DataFrame(columns=columns)

for i, benchmark in enumerate(benchmarks):
    trace_file = f"{base_dir}/{sub_dir}/{benchmark}_{platform}_6x6.csv"
    
    if not os.path.exists(trace_file):
        continue

    df = pd.read_csv(
        trace_file,
        skipinitialspace=True,
    )

    wg_finish_time = df[df['what']=='avg_last_WG_finish_time']

    high_central_avg = 0
    low_central_avg = 0
    edge_avg = 0
    corner_avg = 0

    for c in range(num_chiplets):
        wg_finish_time_per_chiplet = \
            wg_finish_time[wg_finish_time['where']==f'chiplet-{c}']['value'].item()
        
        if c in high_central:
            high_central_avg += wg_finish_time_per_chiplet
        elif c in low_central:
            low_central_avg += wg_finish_time_per_chiplet
        elif c in edge:
            edge_avg += wg_finish_time_per_chiplet
        elif c in corner:
            corner_avg += wg_finish_time_per_chiplet

    high_central_avg /= len(high_central)
    low_central_avg /= len(low_central)
    edge_avg /= len(edge)
    corner_avg /= len(corner)

    result_df.loc[i] = {
        'benchmark': benchmark,
        'high_central': high_central_avg,
        'low_central': low_central_avg,
        'edge': edge_avg,
        'corner': corner_avg,
    }

result_df.to_csv(f"{base_dir}/{sub_dir}/central_edge_wg_time.csv", index=False)

result_df

,benchmark,high_central,low_central,edge,corner
0,fdtd2d,0.000062,0.000062,0.000068,0.000075
1,jacobi1d,0.000105,0.000078,0.000100,0.000105
2,fastwalshtransform,0.000221,0.000177,0.000218,0.000225
3,relu,0.000715,0.000506,0.000626,0.000604
4,fft,0.000239,0.000204,0.000248,0.000259
5,fir,0.000131,0.000120,0.000140,0.000160
6,sssp,0.000070,0.000053,0.000064,0.000074
7,convolution2d,0.000253,0.000197,0.000232,0.000240
8,color,0.000227,0.000157,0.000194,0.000187
9,bitonicsort,0.000182,0.000128,0.000158,0.000154


In [27]:
high_central = [14, 25, 20, 21]
low_central = [7,8,9,10,
                13,16,
                19,22,
                25,28]
edge = [1,2,3,4,
        6,12,18,24,
        11,17,23,29,
        31,32,33,34]
corner = [0, 5, 30, 35]

columns = ['benchmark', 
           'high_central',
           'low_central',
           'edge',
           'corner',
           ]

result_df = pd.DataFrame(columns=columns)

for i, benchmark in enumerate(benchmarks):
    report_file = f"{report_dir}/{benchmark}_{platform}_6x6.csv"

    if not os.path.exists(report_file):
        print(f"No {report_file} trace file found")
        continue
    
    df = pd.read_csv(
        report_file,
        skipinitialspace=True,
    )

    high_central_avg = 0
    low_central_avg = 0
    edge_avg = 0
    corner_avg = 0

    for c in range(num_chiplets):
        hop_count = df[df['what']==f'total-hop-count-per-chiplet-{c}']['value'].item()
        rdma_count = df[df['what']==f'total-rdma-count-per-chiplet-{c}']['value'].item()

        if rdma_count == 0:
            continue

        if c in high_central:
            high_central_avg += (hop_count/rdma_count)
        elif c in low_central:
            low_central_avg += (hop_count/rdma_count)
        elif c in edge:
            edge_avg += (hop_count/rdma_count)
        elif c in corner:
            corner_avg += (hop_count/rdma_count)

    high_central_avg /= len(high_central)
    low_central_avg /= len(low_central)
    edge_avg /= len(edge)
    corner_avg /= len(corner)

    result_df.loc[i] = {
        'benchmark': benchmark,
        'high_central': high_central_avg,
        'low_central': low_central_avg,
        'edge': edge_avg,
        'corner': corner_avg,
    }

# result_df.to_csv(f"{report_dir}/central_edge_hop_count.csv", index=False)
result_df

No /home/ehdgus7279/mgvm/results/reports/mesh/gups_mesh_6x6.csv trace file found


,benchmark,high_central,low_central,edge,corner
0,fdtd2d,1.002432,0.902228,2.319190,2.259897
1,jacobi1d,1.007215,0.908270,2.257721,4.510600
2,fastwalshtransform,5.000000,4.200000,3.040176,5.546838
3,relu,1.007876,0.913198,2.258956,4.537240
4,fft,2.458065,2.395828,2.914260,4.214867
5,fir,1.006901,0.911790,2.260492,4.511065
6,sssp,1.082665,1.966488,2.387044,5.512375
7,convolution2d,1.000773,0.900753,2.256174,4.500470
8,color,1.133228,1.059885,2.476384,4.840316
9,bitonicsort,1.016660,0.912968,2.262145,4.514679
